## Example: Sclareol Cell Factory Design using ecFactory

This notebook demonstrates how to use the **ecFactory** algorithm to design a *S. cerevisiae* cell factory for sclareol production.

Sclareol is a diterpene labdanoid compound with applications in perfumery and as a pharmaceutical precursor. The heterologous biosynthetic pathway consists of:
- **FRTT** (farnesyltranstransferase): FPP + IPP → GGPP
- **SsLPPS** (labdenyl-PP synthase from *Salvia sclarea*): GGPP → LPP
- **SsTPS** (sclareol synthase from *Salvia sclarea*): LPP → sclareol

The ethanol-fed batch fermentation conditions are modeled based on experimental data from two engineered strains:
- **CXM18**: base sclareol-producing strain (μ = 0.16 h⁻¹, q_EtOH = 14.8 mmol/gDW/h)
- **SCX42**: improved strain (μ = 0.18 h⁻¹, q_EtOH = 20 mmol/gDW/h)

Two model types are demonstrated:
1. **ecGEM** (enzyme-constrained GEM): `sclareol_ecYeastGEM_batch.xml`
2. **EFL** (enzyme-constrained + expression model): `sclareol_cEFL.json`

In [ ]:
import sys
import cobra
from strainOptimizer import strainOptimizer_engine, WorkflowParameters

# cobra.Configuration().tolerance = 1e-9

### (Optional) Add heterologous sclareol pathway

The models used in this notebook already contain the sclareol biosynthesis pathway. If you need to add the pathway to a base ecGEM or EFL model, refer to:
- [`0.add_sclareol_pathway.py`](0.add_sclareol_pathway.py) — adds FRTT, SsLPPS, SsTPS reactions to the GAN ecYeast model

The demand reaction `DM_sclareol_c` serves as the optimization target.

### 1. Prepare parameters

`WorkflowParameters` holds all settings in three groups:

- **model**: model file path, type (`ecGEM` / `etfl`), solver, growth reaction ID
- **strain**: target reaction, product name, carbon source reaction, uptake rate, optional measured growth rate
- **algorithm**: design algorithm (`ecFactory`), scanning steps, action thresholds, output settings

Sclareol-specific notes:
- Carbon source is **ethanol** (`r_1761_REV` for ecGEM, `r_1761` for EFL), not glucose
- `substrate_MW` = 0.046 g/mmol (ethanol molecular weight)
- `action_thresholds` = [0.05, 0.5, 1.05]: enzyme usage ratio boundaries for KO / KD / OE classification

In [ ]:
# Experimental phenotype data for sclareol-producing strains
# Keys: ethanol uptake rate (mmol/gDW/h) and measured growth rate (h-1)
strain_phenotype_dict = {
    'CXM18': {
        'c_uptake': 8.2,
        'exp_growth': 0.16,      # measured growth rate (h-1)
    },
    'SCX42': {
        'c_uptake': 11.5,
        'exp_growth': 0.18,
    },
    'normal': {}                 # no phenotype constraints; use 50% max yield as default
}

# Select which strain background to simulate
strain = 'normal'   # options: 'CXM18', 'SCX42', 'normal'
selected_phenotype = strain_phenotype_dict[strain]
print(f"Selected strain: {strain}")
print(f"Phenotype: {selected_phenotype}")

### 2. Run ecFactory design — ecGEM model (ethanol carbon source)

In [ ]:
# Model parameters
model_params = {
    'model_path': 'models/yeast/sclareol_ecYeastGEM_batch.xml',
    'model_type': 'ecGEM',
    'solver': 'optlang-gurobi',
    'growth_id': 'r_2111',
}

# Strain parameters — ethanol as sole carbon source
strain_params = {
    'target_id': 'DM_sclareol_c',
    'product_name': 'sclareol',
    'c_source': 'r_1761_REV',   # ethanol exchange reaction (ecGEM convention: positive uptake)
    'c_uptake': selected_phenotype.get('c_uptake', 10),
    'substrate_MW': 0.046,       # g/mmol ethanol
}

# Provide measured growth rate if available (used to set scanning range)
if 'exp_growth' in selected_phenotype:
    strain_params['experimental_yield'] = (
        selected_phenotype['exp_growth']
        / (strain_params['c_uptake'] * strain_params['substrate_MW'])
    )

# Algorithm parameters
algorithm_params = {
    'design_algorithm': 'ecFactory',
    'simulation_method': 'ppfba',
    'experimental_yield': strain_params.pop('experimental_yield', None),
    'steps': 123,
    'action_thresholds': [0.05, 0.5, 1.05],
    'remove_essential': True,
    'calculate_fcc': True,
    'save_results': True,
    'output_directory': f'./results/sclareol_{strain}',
}

params_ecgem = WorkflowParameters(
    model=model_params,
    strain=strain_params,
    algorithm=algorithm_params,
)

print(f"Product      : {params_ecgem.strain['product_name']}")
print(f"Target rxn   : {params_ecgem.strain['target_id']}")
print(f"Carbon source: {params_ecgem.strain['c_source']}  (ethanol)")
print(f"Model type   : {params_ecgem.model['model_type']}")
print(f"Algorithm    : {params_ecgem.algorithm['design_algorithm']}")
print(f"Strain       : {strain}")

In [ ]:
# Create engine and load model
engine_ecgem = strainOptimizer_engine(params_ecgem)
model_ecgem = engine_ecgem.load_model()

# Suppress glucose uptake so that ethanol is the sole carbon source
model_ecgem.reactions.get_by_id('r_1714_REV').bounds = (0, 0)

# Run ecFactory
import time
t0 = time.time()
final_result_ecgem = engine_ecgem.run_design()
print(f"\nCompleted in {time.time() - t0:.1f} s")

In [ ]:
summary_ecgem = engine_ecgem.get_results_summary()
summary_ecgem

#### Parse result

The combined result DataFrame contains:

| Column | Description |
|--------|-------------|
| `gene_name` | Gene name (SGD) |
| `k_score` | ecFSEOF score — higher = stronger flux coupling to sclareol |
| `action` | Recommended modification: **OE** overexpress, **KD** knockdown, **KO** knockout |
| `EUVA_priority_level` | Priority from enzyme usage variability analysis (1 = highest) |
| `minimal_candidates_set` | 1 if member of the minimal target set needed for max yield |

In [ ]:
final_result_ecgem.head(15)

In [ ]:
result_l1 = engine_ecgem.all_results['level 1 result']
result_l2 = engine_ecgem.all_results['level 2 result']
result_l3 = engine_ecgem.all_results['level 3 result']
result_fcc = engine_ecgem.all_results.get('fcc_filtered_result', None)

print(f"Level 1 (ecFSEOF candidates) : {len(result_l1)} genes")
print(f"Level 2 (EUVA filtered)       : {len(result_l2)} genes")
print(f"Level 3 (minimal set)         : {len(result_l3)} genes")
if result_fcc is not None:
    print(f"FCC filtered result           : {len(result_fcc)} genes")

In [ ]:
print("Action summary (combined result):")
print(final_result_ecgem['action'].value_counts().to_string())

In [ ]:
print("EUVA priority level distribution:")
print(final_result_ecgem['EUVA_priority_level'].value_counts().to_string())

In [ ]:
# High-confidence targets: level 1 and 2 EUVA priority, with FCC if available
high_confidence = final_result_ecgem[
    final_result_ecgem['EUVA_priority_level'].isin([1, 2])
].sort_values('EUVA_priority_level')

print(f"High-confidence targets (Level 1+2): {len(high_confidence)}")
high_confidence

In [ ]:
minimal_set = final_result_ecgem[
    final_result_ecgem['minimal_candidates_set'] == 1
]
print(f"Minimal candidate set for maximum sclareol yield: {len(minimal_set)} genes")
minimal_set

### 3. Run ecFactory design — EFL model (enzyme + expression constraints)

The EFL (enzyme and flux limitation) model integrates enzyme expression and resource allocation constraints for more realistic predictions. Note that EFL simulations require **CPLEX** or **Gurobi** — CPLEX is generally more stable for this model type.

In [ ]:
efl_model_params = {
    'model_path': 'models/yeast/sclareol_cEFL.json',
    'model_type': 'etfl',
    'solver': 'optlang-cplex',   # CPLEX recommended for EFL/ETFL models
    # 'solver': 'optlang-gurobi',
    'growth_id': 'r_2111',
}

efl_strain_params = {
    'target_id': 'DM_sclareol_c',
    'product_name': 'sclareol',
    'c_source': 'r_1761',        # ethanol exchange (EFL convention: negative = uptake)
    'c_uptake': selected_phenotype.get('c_uptake', 10),
    'substrate_MW': 0.046,
}

if 'exp_growth' in selected_phenotype:
    efl_exp_yield = (
        selected_phenotype['exp_growth']
        / (efl_strain_params['c_uptake'] * efl_strain_params['substrate_MW'])
    )
else:
    efl_exp_yield = None

efl_algorithm_params = {
    'design_algorithm': 'ecFactory',
    'simulation_method': 'ppfba',
    'experimental_yield': efl_exp_yield,
    'steps': 123,
    'action_thresholds': [0.05, 0.5, 1.05],
    'remove_essential': True,
    'calculate_fcc': False,      # FCC calculation is slow for EFL models
    'save_results': True,
    'output_directory': f'./results/sclareol_{strain}_efl',
}

# Update engine parameters (reuse engine or create new one)
engine_efl = strainOptimizer_engine(
    WorkflowParameters(
        model=efl_model_params,
        strain=efl_strain_params,
        algorithm=efl_algorithm_params,
    )
)

print(f"Model type   : {engine_efl.params.model['model_type']}")
print(f"Carbon source: {engine_efl.params.strain['c_source']}  (ethanol, EFL convention)")

In [ ]:
model_efl = engine_efl.load_model()

# Block glucose uptake
try:
    model_efl.reactions.get_by_id('r_1714').bounds = (0, 0)
except Exception:
    pass

t0 = time.time()
final_result_efl = engine_efl.run_design()
print(f"\nCompleted in {time.time() - t0:.1f} s")

engine_efl.get_results_summary()

In [ ]:
final_result_efl.head(15)

In [ ]:
print("Action summary — EFL model:")
print(final_result_efl['action'].value_counts().to_string())

print("\nEUVA priority level — EFL model:")
print(final_result_efl['EUVA_priority_level'].value_counts().to_string())

### 4. Compare ecGEM and EFL predictions

Targets appearing in both models with consistent actions are the highest-confidence engineering candidates.

In [ ]:
import pandas as pd

# Merge on gene index
ecgem_genes = set(final_result_ecgem.index)
efl_genes   = set(final_result_efl.index)
common_genes = ecgem_genes & efl_genes

print(f"ecGEM targets : {len(ecgem_genes)}")
print(f"EFL targets   : {len(efl_genes)}")
print(f"Common targets: {len(common_genes)}")

df_compare = final_result_ecgem.loc[list(common_genes), ['gene_name', 'action', 'EUVA_priority_level']].copy()
df_compare.columns = ['gene_name', 'action_ecgem', 'priority_ecgem']
df_compare['action_efl'] = final_result_efl.loc[list(common_genes), 'action']
df_compare['action_consistent'] = df_compare['action_ecgem'] == df_compare['action_efl']

consistent = df_compare[df_compare['action_consistent']].sort_values('priority_ecgem')
print(f"\nConsistent targets (same action in both models): {len(consistent)}")
consistent

### 5. Save results to Excel

In [ ]:
import os

os.makedirs('./results', exist_ok=True)

# ecGEM results
ecgem_out = f'./results/sclareol_ecgem_ethanol_{strain}_ecFactory_result.xlsx'
with pd.ExcelWriter(ecgem_out) as writer:
    for key, df in engine_ecgem.all_results.items():
        if isinstance(df, pd.DataFrame):
            df.to_excel(writer, sheet_name=key[:31])
print(f"ecGEM results saved → {ecgem_out}")

# EFL results
efl_out = f'./results/sclareol_efl_ethanol_{strain}_ecFactory_result.xlsx'
with pd.ExcelWriter(efl_out) as writer:
    for key, df in engine_efl.all_results.items():
        if isinstance(df, pd.DataFrame):
            df.to_excel(writer, sheet_name=key[:31])
print(f"EFL results saved   → {efl_out}")

# Cross-model comparison
compare_out = f'./results/sclareol_{strain}_ecgem_vs_efl_comparison.xlsx'
df_compare.to_excel(compare_out)
print(f"Comparison saved    → {compare_out}")